# BTS — Etapa B4: K-means original con checkpoints (SageMaker)

Continúa después de B1, B2 y B3. **No recalcula Silver ni la ingeniería de variables de la etapa A**. Reutiliza los mismos Parquet de `features_seed42_h240_chunk50000` y replica el procedimiento de la sección **7.4 K-means (tasa por clúster)** del notebook de modelado original.

**Parámetros conservados:** entrenamiento K-means con 1.000.000 de vuelos, validación y prueba de 1.000.000 cada una, `K_GRID=[10,25,50,100]`, `batch_size=50_000`, `n_init=3`, `random_state=42`, `KM_SMOOTH=50`; el mejor K minimiza el **log loss de validación**. Los 4 candidatos se guardan individualmente para reanudar si se cierra el laboratorio.

La tasa `PRIOR` se obtiene de **todo Silver hasta 2024-12-31**, como en el notebook original (no de la muestra K-means). Se aplican la misma imputación por mediana y estandarización sobre el primer millón de entrenamiento. No se reducen filas ni métricas.

**Ejecuta desde `/home/ec2-user/SageMaker/src/` con kernel Python 3.**

In [1]:
%pip install -q numpy pandas pyarrow scikit-learn duckdb joblib boto3


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 0. Rutas y configuración original

In [2]:
from pathlib import Path
import gc, json, shutil, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import duckdb
import boto3
import joblib
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, average_precision_score, roc_auc_score
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
assert ROOT.name == "src", f"Abre este notebook desde SageMaker/src, ahora: {ROOT}"
PARTS_BASE = Path("../data/checkpoints/features_seed42_h240_chunk50000")
B1_DIR = Path("../outputs/outputs_modeling/checkpoints/lightgbm")
OUT_DIR = Path("../outputs/outputs_modeling/checkpoints/kmeans")
TRIAL_DIR = OUT_DIR / "trials"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

BUCKET = "bts-flights-analysis"
B1_PREFIX = "modeling/checkpoints/lightgbm-original-seed42-h240"
S3_PREFIX = "modeling/checkpoints/kmeans-original-seed42-h240"
SEED, TARGET = 42, "DepDel15"
N_TRAIN, N_EVAL, N_KMEANS = 4_000_000, 1_000_000, 1_000_000
K_GRID, KM_SMOOTH = [10, 25, 50, 100], 50
COST_FP, COST_FN = 1.0, 3.0
TRAIN_END, VAL_END, PRED_HORIZON_MIN = "2024-12-31", "2025-08-31", 240
SILVER_GLOB = "../data/silver/flights/**/*.parquet"

print("Etapa A:", PARTS_BASE.resolve())
print("Salida B4:", OUT_DIR.resolve())
print("Disco libre:", round(shutil.disk_usage(ROOT).free / 1024**3, 2), "GiB")
print("scikit-learn:", sklearn.__version__)

Etapa A: /home/ec2-user/SageMaker/data/checkpoints/features_seed42_h240_chunk50000
Salida B4: /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/kmeans
Disco libre: 25.45 GiB
scikit-learn: 1.7.2


## 1. Comprobar los 120 Parquet originales (sin cargar sus filas)

In [3]:
PARTS = {}
for split, expected in {"train": N_TRAIN, "val": N_EVAL, "test": N_EVAL}.items():
    folder = PARTS_BASE / split
    paths = sorted(folder.glob("part_*.parquet"))
    assert (folder / "_SUCCESS").exists(), f"Falta _SUCCESS en {folder}"
    assert paths, f"No se encontraron Parquet en {folder}"
    count = sum(pq.ParquetFile(f).metadata.num_rows for f in paths)
    assert count == expected, f"{split}: {count:,} filas, esperadas {expected:,}"
    PARTS[split] = paths
    print(f"✔ {split}: {len(paths)} archivos, {count:,} filas")
assert len(PARTS["train"]) == 80 and len(PARTS["val"]) == 20 and len(PARTS["test"]) == 20

✔ train: 80 archivos, 4,000,000 filas
✔ val: 20 archivos, 1,000,000 filas
✔ test: 20 archivos, 1,000,000 filas


## 2. Misma lista de variables del K-means original

Se utilizan únicamente variables numéricas; **no se incluye `DepDel15` entre las variables usadas para crear los clústeres**.

In [4]:
HIST_KEYS = ["origin", "route", "airline", "tail", "origin_hr"]
HIST_COLS = [f"{k}_rate_{s}" for k in HIST_KEYS for s in ("all", "30d")]
SCHED_COLS = ["origin_dep_load", "origin_day_load", "dest_arr_load", "tail_leg_num", "turnaround_min"]
STATUS_COLS = ["prev_leg_delayed", "tail_delayed_known", "tail_known_legs", "origin_known_today", "origin_delay_rate_today"]
KM_COLS = (["Distance", "CRSElapsedTime", "dep_min", "arr_min", "days_to_holiday", "is_weekend"]
           + HIST_COLS + SCHED_COLS + STATUS_COLS)
COLS_TO_READ = KM_COLS + [TARGET]
missing = set(COLS_TO_READ) - set(pq.ParquetFile(PARTS["train"][0]).schema_arrow.names)
assert not missing, f"Faltan columnas de la etapa A: {sorted(missing)}"
assert len(KM_COLS) == len(set(KM_COLS)), "Hay características repetidas"

(OUT_DIR / "feature_schema.json").write_text(json.dumps({
    "feature_names": KM_COLS,
    "numeric_cols": KM_COLS,
    "categorical_cols": [],
    "prediction_horizon_min": PRED_HORIZON_MIN,
    "preprocessor": "SimpleImputer(strategy=median) + StandardScaler()",
}, indent=2, ensure_ascii=False))
print("✔ Columnas originales de K-means:", len(KM_COLS))

# Recupera B1 solo si los NPZ no siguen en disco (por ejemplo, tras reiniciar el laboratorio).
def check_b1_labels(split, y):
    path = B1_DIR / f"{split}_predictions.npz"
    if not path.exists():
        B1_DIR.mkdir(parents=True, exist_ok=True)
        session = boto3.Session()
        if session.get_credentials() is None:
            raise RuntimeError("Faltan credenciales AWS para recuperar las etiquetas B1; inicia el Learner Lab")
        session.client("s3").download_file(BUCKET, f"{B1_PREFIX}/{split}_predictions.npz", str(path))
        print("Descargado B1:", path)
    with np.load(path) as z:
        expected = z["y"]
        assert y.shape == expected.shape and np.array_equal(y, expected), (
            f"Las etiquetas de {split} no coinciden fila por fila con B1. Detener la comparación.")
    print(f"✔ {split}: etiquetas alineadas con B1")


def load_numeric(split, max_rows=None):
    """Lee solo las 26 variables numéricas y etiqueta, preservando el orden Parquet."""
    t0 = time.time()
    arrays, labels = [], []
    remaining = max_rows
    for i, path in enumerate(PARTS[split], 1):
        if remaining is not None and remaining <= 0:
            break
        frame = pd.read_parquet(path, columns=COLS_TO_READ)
        if remaining is not None and len(frame) > remaining:
            frame = frame.iloc[:remaining]
        numeric = frame[KM_COLS].apply(pd.to_numeric, errors="coerce")
        arrays.append(numeric.to_numpy(dtype=np.float64, na_value=np.nan, copy=True))
        labels.append(frame[TARGET].to_numpy(dtype=np.uint8, copy=True))
        if remaining is not None:
            remaining -= len(frame)
        del frame, numeric
        if i % 20 == 0:
            print(f"  {split}: {i} fragmentos leídos")
    X = np.concatenate(arrays, axis=0)
    y = np.concatenate(labels, axis=0)
    del arrays, labels
    gc.collect()
    if max_rows is not None:
        assert len(y) == max_rows, f"{split}: esperadas {max_rows:,} filas, obtenidas {len(y):,}"
    print(f"✔ {split}: {len(y):,} filas, {X.shape[1]} variables, {X.nbytes/1024**3:.2f} GiB, {time.time()-t0:.0f}s")
    return X, y

✔ Columnas originales de K-means: 26


## 3. Recuperar la tasa `PRIOR` original desde Silver completo

El notebook original la calcula con `AVG(DepDel15)` sobre **todos los vuelos anteriores o iguales a `TRAIN_END`**, no sobre las muestras. Guardamos ese valor localmente para no repetir el escaneo tras reinicios.

In [5]:
PRIOR_PATH = OUT_DIR / "prior.json"
if PRIOR_PATH.exists():
    PRIOR = float(json.loads(PRIOR_PATH.read_text())["prior"])
    print("PRIOR reutilizado:", PRIOR)
else:
    t0 = time.time()
    DB_PATH = Path("../data/checkpoints/modeling.duckdb")
    if DB_PATH.exists():
        con = duckdb.connect(str(DB_PATH), read_only=True)
        con.execute("SET threads = 2")
        con.execute("SET memory_limit = '2GB'")
        try:
            PRIOR = float(con.execute(
                f"SELECT AVG({TARGET}) FROM base WHERE FlightDate <= DATE '{TRAIN_END}'"
            ).fetchone()[0])
        finally:
            con.close()
    else:
        con = duckdb.connect()
        con.execute("SET threads = 2")
        con.execute("SET memory_limit = '2GB'")
        try:
            PRIOR = float(con.execute(f"""
                SELECT AVG({TARGET}) FROM read_parquet('{SILVER_GLOB}',
                    hive_partitioning=true, hive_types={{'Year':'SMALLINT','Month':'TINYINT'}})
                WHERE FlightDate <= DATE '{TRAIN_END}'
            """).fetchone()[0])
        finally:
            con.close()
    assert 0 < PRIOR < 1, f"PRIOR inesperado: {PRIOR}"
    PRIOR_PATH.write_text(json.dumps({"prior": PRIOR, "source": "all silver flights before train_end inclusive",
                                      "train_end": TRAIN_END}, indent=2))
    print(f"PRIOR calculado: {PRIOR:.6f}, {time.time()-t0:.0f}s")

PRIOR calculado: 0.208353, 0s


## 4. Ajustar el mismo preprocesamiento sobre el primer millón de train

Equivale a `km_pre.fit_transform(train[KM_COLS].iloc[:N_KMEANS])` del notebook original. Guarda el preprocesamiento en disco para poder reanudar.

In [6]:
PRE_PATH = OUT_DIR / "preprocessor.joblib"
X_train_raw, y_train = load_numeric("train", max_rows=N_KMEANS)
if PRE_PATH.exists():
    km_pre = joblib.load(PRE_PATH)
    Xk_tr = km_pre.transform(X_train_raw)
    print("Preprocesamiento previo recuperado")
else:
    km_pre = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    Xk_tr = km_pre.fit_transform(X_train_raw)
    joblib.dump(km_pre, PRE_PATH)
    print("Preprocesamiento ajustado y guardado")
assert Xk_tr.shape == (N_KMEANS, len(KM_COLS)), "Dimensión de entrenamiento inesperada"
del X_train_raw
gc.collect()
print("✔ Matriz K-means train:", Xk_tr.shape, "dtype:", Xk_tr.dtype)

  train: 20 fragmentos leídos
✔ train: 1,000,000 filas, 26 variables, 0.19 GiB, 1s
Preprocesamiento ajustado y guardado
✔ Matriz K-means train: (1000000, 26) dtype: float64


## 5. Preparar validación y comprobar alineación con B1

In [7]:
X_val_raw, y_val = load_numeric("val")
check_b1_labels("val", y_val)
Xk_va = km_pre.transform(X_val_raw)
assert Xk_va.shape == (N_EVAL, len(KM_COLS))
del X_val_raw
gc.collect()
print("✔ Matriz de validación:", Xk_va.shape)

  val: 20 fragmentos leídos
✔ val: 1,000,000 filas, 26 variables, 0.19 GiB, 1s
✔ val: etiquetas alineadas con B1
✔ Matriz de validación: (1000000, 26)


## 6. Entrenar los cuatro K originales y guardar cada resultado

**Se puede reanudar:** cuando termina cada valor de K, el ajuste y sus probabilidades de validación se guardan en `trials/`. En una nueva sesión, si están completos, se reutilizan en lugar de entrenar otra vez. Para cada K se evalúa exactamente con las mismas métricas de validación que el notebook original.

A diferencia del original, no se predice prueba para todos los K: se predice solamente para el ganador elegido mediante validación. Eso evita memoria y CPU innecesarios sin afectar la selección ni la evaluación final.

In [8]:
def cluster_rates(labels, y, k):
    """Misma función de suavizado que la sección original 7.4."""
    g = (pd.DataFrame({"c": labels, "y": y})
         .groupby("c").y.agg(["sum", "count"]).reindex(range(k), fill_value=0))
    return ((g["sum"] + KM_SMOOTH * PRIOR) / (g["count"] + KM_SMOOTH)).values

km_rows = []
for k in K_GRID:
    trial_path = TRIAL_DIR / f"kmeans_k{k}.joblib"
    val_path = TRIAL_DIR / f"val_k{k}.npz"
    if trial_path.exists() and val_path.exists():
        with np.load(val_path) as z:
            p_va = z["p"]
            assert np.array_equal(z["y"], y_val), f"Se alteró validación del trial K={k}"
        saved = joblib.load(trial_path)
        assert int(saved["k"]) == k and len(saved["cluster_rates"]) == k
        print(f"✔ K={k}: recuperado desde checkpoints")
    else:
        t0 = time.time()
        km = MiniBatchKMeans(n_clusters=k, batch_size=50_000, n_init=3, random_state=SEED)
        with threadpool_limits(limits=4):
            km.fit(Xk_tr)
        rate = cluster_rates(km.labels_, y_train, k)
        with threadpool_limits(limits=4):
            p_va = rate[km.predict(Xk_va)]
        # Guardar un K a la vez para sobrevivir a cierres del Learner Lab.
        joblib.dump({"k": k, "kmeans": km, "cluster_rates": rate}, trial_path)
        np.savez_compressed(val_path, y=y_val, p=p_va)
        print(f"✔ K={k}: entrenado y guardado en {time.time()-t0:.0f}s")
        del km, rate
        gc.collect()
    km_rows.append({
        "k": k,
        "log_loss_val": float(log_loss(y_val, p_va)),
        "pr_auc_val": float(average_precision_score(y_val, p_va)),
        "roc_auc_val": float(roc_auc_score(y_val, p_va)),
    })
    del p_va
    gc.collect()

km_table = pd.DataFrame(km_rows).set_index("k")
km_table.to_csv(OUT_DIR / "kmeans_grid_validation.csv")
best_k = int(km_table.log_loss_val.idxmin())
print("\n✔ K elegido por menor log loss en VALIDACIÓN:", best_k)
display(km_table)

✔ K=10: entrenado y guardado en 2s
✔ K=25: entrenado y guardado en 3s
✔ K=50: entrenado y guardado en 4s
✔ K=100: entrenado y guardado en 6s

✔ K elegido por menor log loss en VALIDACIÓN: 100


,log_loss_val,pr_auc_val,roc_auc_val
k,,,
10,0.499546,0.353920,0.647684
25,0.494407,0.378069,0.672516
50,0.492808,0.388288,0.678992
100,0.490664,0.398294,0.682811


## 7. Persistir el mejor K y sus probabilidades de validación

Se conserva **exactamente** la regla de selección original por `log_loss_val`, y se empaquetan el K-means, las tasas por clúster y el preprocesador para poder reproducir la predicción.

In [9]:
best_trial = joblib.load(TRIAL_DIR / f"kmeans_k{best_k}.joblib")
best_km = best_trial["kmeans"]
best_rates = best_trial["cluster_rates"]
with np.load(TRIAL_DIR / f"val_k{best_k}.npz") as z:
    assert np.array_equal(z["y"], y_val)
    np.savez_compressed(OUT_DIR / "val_predictions.npz", y=y_val, p=z["p"])

joblib.dump({
    "preprocessor": km_pre,
    "kmeans": best_km,
    "cluster_rates": best_rates,
    "feature_names": KM_COLS,
    "prior": PRIOR,
    "k": best_k,
    "seed": SEED,
}, OUT_DIR / "model.joblib")
config = {
    "model": "K-means (tasa por clúster)", "seed": SEED,
    "train_rows_available": N_TRAIN, "fit_rows": N_KMEANS,
    "val_rows": N_EVAL, "test_rows": N_EVAL,
    "train_end": TRAIN_END, "val_end": VAL_END,
    "prediction_horizon_min": PRED_HORIZON_MIN,
    "k_grid": K_GRID, "selected_k": best_k,
    "kmeans_params": {"batch_size": 50_000, "n_init": 3, "random_state": SEED},
    "km_smooth": KM_SMOOTH, "prior_from_all_silver_train": float(PRIOR),
    "selection_metric": "minimum log_loss on validation",
    "preprocessing": "SimpleImputer(strategy=median), StandardScaler",
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
print("✔ Modelo seleccionado y artefactos guardados")
# No necesitamos el millón de entrenamiento ni de validación en RAM para predecir prueba.
del Xk_tr, Xk_va, y_train, y_val, km_pre
gc.collect()

✔ Modelo seleccionado y artefactos guardados


55

## 8. Predecir el millón de prueba con K seleccionado, sin reentrenar

El conjunto test **no interviene en la selección de K**. Se utiliza únicamente para evaluación.

In [10]:
X_test_raw, y_test = load_numeric("test")
check_b1_labels("test", y_test)
with threadpool_limits(limits=4):
    # El preprocesamiento y el modelo son exactamente los guardados en el artefacto.
    best_bundle = joblib.load(OUT_DIR / "model.joblib")
    Xk_te = best_bundle["preprocessor"].transform(X_test_raw)
    p_test = best_bundle["cluster_rates"][best_bundle["kmeans"].predict(Xk_te)]
np.savez_compressed(OUT_DIR / "test_predictions.npz", y=y_test, p=p_test)
print(f"✔ Test: {len(y_test):,} predicciones; probabilidad media={float(p_test.mean()):.5f}")
del X_test_raw, Xk_te, y_test, p_test, best_bundle, best_km, best_rates, best_trial
gc.collect()

  test: 20 fragmentos leídos
✔ test: 1,000,000 filas, 26 variables, 0.19 GiB, 1s
✔ test: etiquetas alineadas con B1
✔ Test: 1,000,000 predicciones; probabilidad media=0.21887


29

## 9. Calcular las mismas métricas de B2 y B3 (corrección NumPy → JSON incluida)

Los umbrales se eligen usando **validación** y se aplican sin cambios a **prueba**.

In [11]:
sys.path.insert(0, str(Path("utils").resolve()))
import model_eval as me

with np.load(OUT_DIR / "val_predictions.npz") as z:
    y_val, p_val = z["y"], z["p"]
with np.load(OUT_DIR / "test_predictions.npz") as z:
    y_test, p_test = z["y"], z["p"]

threshold_val = float(me.best_cost_threshold(y_val, p_val, c_fp=COST_FP, c_fn=COST_FN))
metrics = {
    "val": me.evaluate_model("K-means (tasa por clúster)", y_val, p_val, threshold_val,
                             c_fp=COST_FP, c_fn=COST_FN),
    "test": me.evaluate_model("K-means (tasa por clúster)", y_test, p_test, threshold_val,
                              c_fp=COST_FP, c_fn=COST_FN),
    "threshold_from_validation": threshold_val,
    "threshold_theoretical": float(COST_FP / (COST_FP + COST_FN)),
    "selected_k": best_k,
}

def numpy_to_json(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f"Tipo no serializable: {type(obj).__name__}")

(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False,
                                                 default=numpy_to_json))
fields = ("roc_auc", "pr_auc", "precision", "recall", "f1", "brier", "log_loss", "costo_x1000")
print(json.dumps({s: {k: d[k] for k in fields} for s, d in metrics.items() if s in ("val", "test")},
                 indent=2, ensure_ascii=False, default=numpy_to_json))
required = ["model.joblib", "preprocessor.joblib", "feature_schema.json", "run_config.json",
            "prior.json", "val_predictions.npz", "test_predictions.npz", "metrics.json",
            "kmeans_grid_validation.csv"]
assert all((OUT_DIR / f).exists() for f in required), "Faltan artefactos B4"
print("✔ B4 finalizada: modelo, probabilidades, métricas y 4 pruebas de K guardadas")
del y_val, y_test, p_val, p_test
gc.collect()

{
  "val": {
    "roc_auc": 0.682810526100381,
    "pr_auc": 0.398293897145059,
    "precision": 0.3924664200895464,
    "recall": 0.4526411739349091,
    "f1": 0.420411465653503,
    "brier": 0.15835693135799253,
    "log_loss": 0.49066393559545624,
    "costo_x1000": 521.7420000000001
  },
  "test": {
    "roc_auc": 0.6921709136921268,
    "pr_auc": 0.4183716903797733,
    "precision": 0.4027918433362639,
    "recall": 0.473600649024936,
    "f1": 0.4353357202137768,
    "brier": 0.1599777797789874,
    "log_loss": 0.4940678139802911,
    "costo_x1000": 523.048
  }
}
✔ B4 finalizada: modelo, probabilidades, métricas y 4 pruebas de K guardadas


62

## 10. Respaldo B4 en S3

Ejecuta esta celda **solo si las métricas y todos los archivos están guardados**. Se crea el cliente de S3 en este momento para recuperar las credenciales actuales del Learner Lab. Si falla por credenciales, inicia de nuevo el laboratorio y ejecuta únicamente esta celda: **no necesitas entrenar otra vez**.

In [12]:
required = ["model.joblib", "preprocessor.joblib", "feature_schema.json", "run_config.json",
            "prior.json", "val_predictions.npz", "test_predictions.npz", "metrics.json",
            "kmeans_grid_validation.csv"]
assert all((OUT_DIR / f).exists() for f in required), "B4 incompleta; no se subirá"
assert all((TRIAL_DIR / f"kmeans_k{k}.joblib").exists() and
           (TRIAL_DIR / f"val_k{k}.npz").exists() for k in K_GRID), "Faltan pruebas de K"
session = boto3.Session(region_name="us-east-1")
assert session.get_credentials() is not None, "Sin credenciales AWS: inicia o reinicia el Learner Lab"
s3 = session.client("s3")
for path in sorted(OUT_DIR.rglob("*")):
    if not path.is_file():
        continue
    relative = path.relative_to(OUT_DIR).as_posix()
    key = f"{S3_PREFIX}/{relative}"
    s3.upload_file(str(path), BUCKET, key)
    print("Subido:", f"s3://{BUCKET}/{key}")
print("✔ Checkpoint B4 de K-means respaldado en S3")

Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/feature_schema.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/kmeans_grid_validation.csv
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/metrics.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/model.joblib
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/preprocessor.joblib
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/prior.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/run_config.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/test_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/trials/kmeans_k10.joblib
Subido: s3://bts-flights-analysis/modeling/checkpoints/kmeans-original-seed42-h240/tria

## Después de B4

Con B1–B4 terminados, se reconstruirán los cuadros y gráficos del notebook original (`comparison_threshold_cost.csv`, `comparison_threshold_common.csv`, `comparison_alert_budget.csv`, bootstrap pareado, etc.) leyendo **solo los archivos `val_predictions.npz` y `test_predictions.npz`** de cada modelo. Las ablaciones, pesos de clase y diagnóstico de deriva siguen pendientes; los abordaremos por separado para no comprometer la RAM.

**Nota metodológica:** la selección de K usa validación, pero el **veredicto final entre algoritmos** necesita tratar el conjunto de prueba como evaluación final y no como conjunto de selección de hiperparámetros; documentaremos esta distinción en la etapa de comparación.